# Day 26 — DENSE Baseline Reproduction

## Goal

Today we stop reading and start reproducing.

**Target:** get the smallest end-to-end DENSE baseline run on Cora far enough to identify the first real runtime blocker, then continue until we obtain one baseline result.

```text
Environment
    ↓
Dataset
    ↓
Import / data smoke test
    ↓
Minimal DENSE run
    ↓
Record first real error
    ↓
Fix one blocker at a time
    ↓
Baseline result
```

### Rule for today

Do **not** modify the method before the first run. First reproduce the repository behavior exactly enough to observe the real failure point.


## 1. Record the exact repository version

On the Linux GPU machine, clone the official repository separately from your own `research-preparation` repo.

```bash
git clone https://github.com/YushengZhao/bundle-neurips25.git
cd bundle-neurips25
git status
git rev-parse HEAD
```

- Commit hash:
> 80834ed51c5ed25150ebf6535662d33886fc84ad

- Branch:
> master

- Any local modifications before reproduction?
> No

### Q1
Why record the commit hash?

> Because the repository may change over time. The commit hash records the exact code version used for reproduction and makes the experiment reproducible.

## 2. GPU / OS preflight

```bash
nvidia-smi
uname -a
```

- GPU:
> NVIDIA GeForce RTX 4080, 32 GB VRAM

- Driver:
> 595.71.05

- CUDA shown by `nvidia-smi`:
> 13.2

- OS:
> Ubuntu Linux, kernel 5.15.0-78-generic, x86_64

### Q2
Does the machine provide a usable NVIDIA GPU?

> Yes. The machine provides a usable NVIDIA GeForce RTX 4080 with 32 GB VRAM, and nvidia-smi works correctly.


## 3. Create the isolated environment

The official README specifies Linux, Python 3.8.18, PyTorch 2.1.2 and CUDA 12.1.

```bash
conda create -n bundle python==3.8.18 -y
conda activate bundle

conda install pytorch==2.1.2 torchvision==0.16.2 torchaudio==2.1.2 pytorch-cuda=12.1 -c pytorch -c nvidia -y

pip install pyg_lib==0.3.1+pt21cu121 -f https://data.pyg.org/whl/torch-2.1.0+cu121.html
pip install torch_scatter==2.1.2 -f https://data.pyg.org/whl/torch-2.1.0+cu121.html
pip install torch_sparse==0.6.18+pt21cu121 -f https://data.pyg.org/whl/torch-2.1.0+cu121.html
pip install torch_cluster==1.6.3+pt21cu121 -f https://data.pyg.org/whl/torch-2.1.0+cu121.html
pip install torch_spline_conv==1.2.2+pt21cu121 -f https://data.pyg.org/whl/torch-2.1.0+cu121.html
pip install transformers==4.46.3
pip install sentence_transformers==2.2.2
pip install dgl==2.4.0+cu121 -f https://data.dgl.ai/wheels/torch-2.1/cu121/repo.html
pip install openai
pip install torch_geometric==2.5.0
pip install protobuf
pip install accelerate
```

Do not install extra packages unless an actual import error requires them.


## 4. Verify the environment before touching the dataset

```bash
python --version

python - <<'PY'
import torch
print("torch:", torch.__version__)
print("cuda available:", torch.cuda.is_available())
print("torch cuda:", torch.version.cuda)
if torch.cuda.is_available():
    print("gpu:", torch.cuda.get_device_name(0))

import torch_geometric
print("pyg:", torch_geometric.__version__)

import dgl
print("dgl:", dgl.__version__)
PY
```

### Q3
What is the first environment mismatch, if any?

> The first compatibility issue was between `sentence-transformers==2.2.2` and the initially installed `huggingface_hub==0.36.2`. `sentence-transformers` expected the deprecated `cached_download` API, which was absent in the newer Hub version. I resolved it by using `huggingface_hub==0.25.2`, which still provides `cached_download` while remaining compatible with the installed `transformers` version.

## 5. Prepare the Cora data

The README points to:

https://huggingface.co/datasets/Graph-COM/Text-Attributed-Graphs

Put the Cora data under:

```text
bundle-neurips25/
└── dataset/
    └── cora/
```

From the current code, Cora needs **at least**:

```text
dataset/cora/processed_data.pt
dataset/cora/raw_texts.pt
dataset/cora/categories.csv
dataset/cora/llmicl_class_aware_x.pt
```

Verify:

```bash
find dataset/cora -maxdepth 1 -type f -printf '%f\n' | sort
```

### Files actually present

```text
categories.csv
llmicl_class_aware_x.pt
processed_data.pt
raw_texts.pt
```

### Q4
Are all four code-visible Cora dependencies present?

> Yes. All four required Cora files are present: processed_data.pt, raw_texts.pt, categories.csv, and llmicl_class_aware_x.pt.


## 6. Data-loading smoke test

Before calling the LLM or training the GNN, test only the loaders.

```bash
python - <<'PY'
from utils import prepare_text, prepare_graph

raw_texts, labels = prepare_text("cora")
graph_data, embs = prepare_graph("cora", "llmicl_class_aware")

print("num raw texts:", len(raw_texts))
print("num labels:", len(labels))
print("graph nodes:", graph_data.num_nodes)
print("embedding shape:", tuple(embs.shape))
print("edge_index shape:", tuple(graph_data.edge_index.shape))
PY
```

### Result

```text
num raw texts: 2708
num label classes: 7
graph nodes: 2708
embedding shape: (2708, 4096)
edge_index shape: (2, 10556)

DATA LOADING SUCCESS
```

### Q5
Did the repo reach successful data loading?

>  Yes, I made it!


## 7. API setup — only after data loading works

Never place the real key in this notebook, README, Git history, or screenshots.

```bash
export OPENAI_API_KEY="YOUR_KEY"
```

Check only whether the variable exists:

```bash
python - <<'PY'
import os
print("OPENAI_API_KEY set:", bool(os.getenv("OPENAI_API_KEY")))
PY
```

### Q6
Is the API key visible to the process?

> No. `OPENAI_API_KEY` is not currently configured. For the initial reproduction, I will first attempt to reuse the authors' released GPT-4o-mini response caches rather than issuing new API requests.

## 8. Code-risk checkpoint before the first DENSE run

Day 25 found several repository inconsistencies:

1. The README uses `--gnn_type gin` for Cora, while the inspected `prepare_model()` implementation did not clearly expose GIN.
2. `solve()` calls `bundle_resample(...)`, but the inspected repository snapshot did not contain its definition.
3. The inspected `ranking` loss branch contains a suspicious `F.cross_entropy(...)` call without an explicit target.
4. The inspected `bundle_optimize()` signature appeared inconsistent with the way it is called from `solve()`.

**Do not repair these yet.**

> ### Q7
Why should we run before editing?

> We should run the original repository first to establish its actual behavior and reproduce any failures before making changes. This separates repository/environment issues from problems introduced by our own modifications, provides a reproducible baseline, and allows each later fix to be justified by a concrete observed error rather than by speculation.

## 9. First minimal runtime attempt

Use a code-supported GNN backbone and avoid refinement on the first smoke run.

```bash
python bundle.py \
  --device 0 \
  --dataset cora \
  --bundle_size 5 \
  --num_samples 5 \
  --sample_criterion neighbor \
  --max_hop 2 \
  --query_type gpt \
  --model deepseek/deepseek-v4.1-flash
  --loss_type average \
  --gnn_type gcn \
  --stages 10 \
  --lr 0.001 \
  --wd 0.001 \
  --repeat 1
```

This is a **debugging smoke run**, not the paper's official Cora result.

Why this version?

- `num_samples=20`: fewer LLM queries.
- `loss_type=average`: first test the simpler bundle-level CE path before the suspicious ranking branch.
- `gnn_type=gcn`: use a backbone we have actually seen implemented.
- `stages=10`: one stage avoids calling the currently missing `bundle_resample()` and tests sampling → LLM query → GNN optimization → evaluation.

### Result category

- [x] Success
- [ ] Environment/import failure
- [ ] Dataset-loading failure
- [ ] API/query failure
- [ ] Model construction failure
- [ ] Loss/training failure
- [ ] Evaluation failure
- [ ] Other

### First real traceback / output

```text
Bundle Query: 100% 5/5
Bundle valid rate: 100.0000%
Bundle class acc: 100.0000%
Accuracy: 0.4022 ± 0.0000
Time: 1.0169 ± 0.0000
```

### Q8
What is the **first real runtime blocker**?

> The first minimal DENSE smoke run completed successfully after resolving
> several repository inconsistencies. The final blockers were stale or
> inconsistent code paths in the repository rather than the dataset or
> environment.
>
> After minimal fixes, the complete pipeline successfully executed:
> bundle sampling → LLM bundle query → bundle-level supervision →
> GNN optimization → evaluation.
>
> The resulting accuracy was 0.4022. This is not yet a reproduction of
> the paper result because only 5 bundles and simplified smoke-test
> settings were used.

## 10. Fix exactly one blocker

### Blocker
> The smoke run reached GNN optimization but crashed because
> `bundle_optimize()` was called with fewer arguments than required by
> its function definition.

### Root cause
> The released repository contains an inconsistent function interface.
> `solve()` calls `bundle_optimize(bundles, bundle_classes, stage)`,
> while the released function definition additionally requires
> `previous_best_metric` and `best_path`.

### Minimal fix
> For the single-stage smoke test, I removed the two unused legacy
> arguments from the `bundle_optimize()` interface and its return path,
> so that the function signature matches the actual call in `solve()`.

### Files changed
> `bundle.py`

### Why is this fix minimal?
> The change only repairs an internal interface mismatch required to
> execute the released training path. It does not change bundle sampling,
> LLM-generated supervision, the GNN forward pass, the average
> bundle-level loss, or the evaluation metric.

## 11. Escalation ladder

```text
Level 0 — imports + CUDA
Level 1 — Cora data loading
Level 2 — bundle sampling
Level 3 — LLM bundle query
Level 4 — single-stage GNN training with average loss
Level 5 — single-stage ranking loss
Level 6 — multi-stage training + refinement/resampling
Level 7 — official Cora configuration
```

### Current level reached
> **Level 4 — single-stage GNN training with average loss**

The complete path has been verified:

Cora data
→ bundle sampling
→ real LLM bundle query
→ bundle-level labels
→ GCN training with average bundle loss
→ test-set evaluation

The next unresolved level is Level 5: ranking loss.


## 12. Official Cora command — do not use until the smoke run works

The README gives:

```bash
python bundle.py --device 0 --dataset cora --bundle_size 5 --num_samples 100 --sample_criterion neighbor --max_hop 2 --query_type gpt --model gpt-4o --loss_type ranking --gnn_type gin --stages 400 100 100 --valid --lr 0.001 --wd 0.001 --resample --repeat 1
```

### Q9
Which differences remain between our smoke run and the official command?

> Several important differences remain:
>
> - The smoke run uses 5 bundles, while the official Cora command uses 100.
> - The smoke run uses `average` loss, while the official command uses `ranking`.
> - The smoke run uses GCN, while the README specifies GIN for Cora.
> - The smoke run trains for only one 10-epoch stage, while the official
>   command uses stages `400 100 100`.
> - The smoke run does not perform refinement/resampling, while the official
>   command enables `--resample`.
> - The smoke run uses DeepSeek-V4.1-Flash through an OpenAI-compatible API,
>   while the official command specifies GPT-4o.
> - Minimal repository fixes were required before the released code could
>   execute end-to-end.
>
> Therefore, the current 0.4022 result is only a smoke-test baseline and
> cannot be directly compared with the paper's official Cora result.

## 13. Baseline result log

- Dataset:
> Cora

- Commit:
> `80834ed51c5ed25150ebf6535662d33886fc84ad`

- GPU:
> NVIDIA GeForce RTX 4080, 32 GB VRAM

- GNN:
> GCN

- Query model:
> `deepseek/deepseek-v4.1-flash`

- Bundle size:
> 5

- Number of bundles:
> 5

- Loss:
> average

- Stages:
> 10

- Test accuracy:
> 0.4022

- Runtime:
> 1.0169 s for the reported training/evaluation timing.
> The external LLM bundle queries took approximately 16–34 s in the
> observed smoke runs and are not represented by this training timer.

- Cache used?
> No released GPT-4o cache was used for this final smoke run.
> Real LLM queries were issued through an OpenAI-compatible API endpoint.

- Any code modifications?
> Yes. Minimal compatibility/runtime fixes were made to the released
> repository so that the single-stage pipeline could execute.
> These changes should be documented separately from the original commit
> before attempting an official reproduction.

## 14. Researcher's reflection

### Q10
Which failure came from the environment, which came from missing data,
and which came from the repository itself?

> The environment produced a dependency compatibility issue between
> `sentence-transformers` and `huggingface_hub`, which was resolved by
> selecting a compatible Hub version.
>
> The dataset itself was not a blocker after the four required Cora files
> were downloaded and verified; data loading completed successfully.
>
> The major runtime blockers came from the released repository itself,
> including inconsistent function interfaces and incomplete/stale code
> paths. API configuration and provider rate limits were additional
> external runtime issues rather than DENSE algorithmic failures.

### Q11
Did the paper-level understanding help you identify the failing module faster?

> Yes. Because I already understood the intended DENSE pipeline as
> bundle sampling → LLM bundle supervision → GNN optimization →
> refinement → evaluation, I could locate each failure within a specific
> module instead of treating the repository as a black box.
>
> For example, once bundle queries completed successfully, I knew that
> the next failure in `bundle_optimize()` belonged to the GNN training
> stage rather than to data loading or LLM querying.

### Q12
What is the smallest code change required to obtain a baseline without
changing DENSE's research idea?

> The smallest required changes are compatibility fixes that restore the
> intended execution path without altering the method itself.
>
> For the successful single-stage baseline, this means repairing the
> inconsistent query/training interfaces while keeping the core method
> unchanged: graph-based bundle sampling, LLM-generated bundle labels,
> bundle-level supervision, GNN optimization, and test evaluation.

### Q13
Which unresolved discrepancy must be fixed before claiming an official reproduction?

> The released implementation must first be reconciled with the paper and
> README configuration.
>
> In particular, the official Cora setup requires ranking loss, GIN,
> multi-stage training, and bundle refinement/resampling, while the current
> smoke run only verifies GCN + average loss + one short stage.
>
> Therefore, the 0.4022 result must not be presented as an official DENSE
> reproduction.

## Day 26 Completion Checklist

- [x] Recorded the exact repository commit
- [x] Verified Linux GPU access
- [x] Created an isolated environment
- [x] Verified PyTorch + CUDA + PyG + DGL
- [x] Prepared Cora data
- [x] Passed the data-loading smoke test
- [x] Configured the API key safely
- [x] Attempted the minimal DENSE smoke run
- [x] Recorded the first real traceback
- [x] Fixed blockers one at a time
- [x] Obtained at least one end-to-end baseline result
- [x] Recorded the exact configuration and result

**Day 26 status: COMPLETE**

The repository has now been executed end-to-end through:

Cora
→ bundle sampling
→ LLM query
→ bundle supervision
→ GNN training
→ test evaluation

The first smoke-test accuracy is **0.4022**.

This is not an official reproduction of the paper result.
The remaining paper–README–implementation discrepancies will be investigated
in Day 27.